In [2]:
import pandas as pd
import numpy as np

Data Exploration

1. Hospital General Information

In [102]:
hospital_general_info = pd.read_csv('Hospital_General_Information.csv')
print(hospital_general_info.shape)
print(hospital_general_info.columns.tolist())
print(hospital_general_info.head())
print("Rows:", len(hospital_general_info))
print("Unique hospitals:", hospital_general_info["Facility ID"].nunique())
print("Duplicate IDs:", hospital_general_info["Facility ID"].duplicated().sum())
print("Missing Facility IDs:", hospital_general_info["Facility ID"].isna().sum())

(5432, 38)
['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Telephone Number', 'Hospital Type', 'Hospital Ownership', 'Emergency Services', 'Meets criteria for birthing friendly designation', 'Hospital overall rating', 'Hospital overall rating footnote', 'MORT Group Measure Count', 'Count of Facility MORT Measures', 'Count of MORT Measures Better', 'Count of MORT Measures No Different', 'Count of MORT Measures Worse', 'MORT Group Footnote', 'Safety Group Measure Count', 'Count of Facility Safety Measures', 'Count of Safety Measures Better', 'Count of Safety Measures No Different', 'Count of Safety Measures Worse', 'Safety Group Footnote', 'READM Group Measure Count', 'Count of Facility READM Measures', 'Count of READM Measures Better', 'Count of READM Measures No Different', 'Count of READM Measures Worse', 'READM Group Footnote', 'Pt Exp Group Measure Count', 'Count of Facility Pt Exp Measures', 'Pt Exp Group Footnote', 'TE Group Measure 

In [103]:
hospital_general_info["Hospital overall rating"].unique()

array(['4', '3', '2', '1', 'Not Available', '5'], dtype=object)

In [104]:
missing = (

    hospital_general_info.isna()

    .mean()

    .sort_values(ascending=False)

)

print(missing)
print(hospital_general_info[

    "Meets criteria for birthing friendly designation"

].value_counts(dropna=False))

TE Group Footnote                                   0.814065
READM Group Footnote                                0.774669
MORT Group Footnote                                 0.742268
Pt Exp Group Footnote                               0.628498
Safety Group Footnote                               0.603829
Meets criteria for birthing friendly designation    0.583395
Hospital overall rating footnote                    0.572717
City/Town                                           0.000000
Count of Safety Measures Worse                      0.000000
READM Group Measure Count                           0.000000
Count of Facility READM Measures                    0.000000
Count of READM Measures Better                      0.000000
Count of READM Measures No Different                0.000000
Count of READM Measures Worse                       0.000000
Count of Safety Measures Better                     0.000000
Address                                             0.000000
Pt Exp Group Measure Cou

In [113]:
hospital = hospital_general_info.copy()
hospital = hospital.drop(columns=[

    "Meets criteria for birthing friendly designation",

    "Telephone Number",

    "Address"

])
hospital = hospital.replace("Not Available", pd.NA)
hospital["Facility ID"] = hospital["Facility ID"].astype("string")
hospital["ZIP Code"] = (

    hospital["ZIP Code"]

    .astype("string")

    .str.zfill(5)

)
hospital["Hospital overall rating"] = pd.to_numeric(

    hospital["Hospital overall rating"],

    errors="coerce"

)
count_cols = [

    col for col in hospital.columns

    if col.startswith("Count of")

]

for col in count_cols:

    hospital[col] = pd.to_numeric(

        hospital[col],

        errors="coerce"

    )

In [114]:
missing_summary = pd.DataFrame({

    "missing_count": hospital.isna().sum(),

    "missing_pct": hospital.isna().mean()

}).sort_values("missing_pct", ascending=False)

print(missing_summary)
print("Rows:", len(hospital))

print("Unique facility IDs:", hospital["Facility ID"].nunique())

print("Duplicate facility IDs:", hospital["Facility ID"].duplicated().sum())
for col in [

    "Hospital Type",

    "Hospital Ownership",

    "Emergency Services"

]:

    print("\n", col)

    print(hospital[col].value_counts(dropna=False))

                                       missing_count  missing_pct
TE Group Footnote                               4422     0.814065
READM Group Footnote                            4208     0.774669
MORT Group Footnote                             4032     0.742268
Pt Exp Group Footnote                           3414     0.628498
Safety Group Footnote                           3280     0.603829
Hospital overall rating footnote                3111     0.572717
Hospital overall rating                         2250     0.414212
Count of Safety Measures Worse                  2078     0.382548
Count of Safety Measures No Different           2078     0.382548
Count of Safety Measures Better                 2078     0.382548
Count of Facility Safety Measures               2078     0.382548
Count of Facility Pt Exp Measures               1957     0.360272
Count of MORT Measures Worse                    1327     0.244293
Count of MORT Measures No Different             1327     0.244293
Count of M

In [115]:
print(hospital["Hospital overall rating"].describe())
mort_sum = (

    hospital["Count of MORT Measures Better"]

    + hospital["Count of MORT Measures No Different"]

    + hospital["Count of MORT Measures Worse"]

)

mort_check = (

    mort_sum

    == hospital["Count of Facility MORT Measures"]

)

print(mort_check.value_counts(dropna=False))
hospital["high_quality"] = (

    hospital["Hospital overall rating"] >= 4

)

count    3182.000000
mean        3.206788
std         1.094115
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: Hospital overall rating, dtype: float64
True     4105
False    1327
Name: count, dtype: int64


In [116]:
hospital["readmission_worse_rate"] = np.where(

    hospital["Count of Facility READM Measures"] > 0,

    hospital["Count of READM Measures Worse"]

    / hospital["Count of Facility READM Measures"],

    np.nan

)

In [121]:
hospital.to_csv(

    "cleaned/hospital_general_clean.csv",

    index=False

)

2. Patient Survey HCAHPS

In [54]:
hcahps = pd.read_csv("HCAHPS-Hospital.csv")
print(hcahps.shape)
print(hcahps.columns.tolist())
print(hcahps.head())


(325856, 22)
['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Telephone Number', 'HCAHPS Measure ID', 'HCAHPS Question', 'HCAHPS Answer Description', 'Patient Survey Star Rating', 'Patient Survey Star Rating Footnote', 'HCAHPS Answer Percent', 'HCAHPS Answer Percent Footnote', 'HCAHPS Linear Mean Value', 'Number of Completed Surveys', 'Number of Completed Surveys Footnote', 'Survey Response Rate Percent', 'Survey Response Rate Percent Footnote', 'Start Date', 'End Date']
  Facility ID                    Facility Name                 Address  \
0      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   
1      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   
2      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   
3      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   
4      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   

  City/Town State  ZIP Code Coun

/var/folders/d5/xf03rj855y5c5nqtgqmn3c380000gn/T/ipykernel_34566/1037150502.py:1: DtypeWarning: Columns (12,14,17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  hcahps = pd.read_csv("HCAHPS-Hospital.csv")


In [55]:
for col in [

    "HCAHPS Measure ID",

    "HCAHPS Question",

    "HCAHPS Answer Description"

]:

    if col in hcahps.columns:

        print("\n", col)

        print(hcahps[col].value_counts(dropna=False).head(30))


 HCAHPS Measure ID
HCAHPS Measure ID
H_COMP_1_A_P                 4792
H_DISCH_HELP_N_P             4792
H_CLEAN_HSP_U_P              4792
H_CLEAN_HSP_SN_P             4792
H_CLEAN_HSP_A_P              4792
H_SYMPTOMS_Y_P               4792
H_SYMPTOMS_N_P               4792
H_DISCH_HELP_Y_P             4792
H_COMP_6_STAR_RATING         4792
H_COMP_1_SN_P                4792
H_COMP_6_LINEAR_SCORE        4792
H_COMP_6_Y_P                 4792
H_COMP_6_N_P                 4792
H_SIDE_EFFECTS_U_P           4792
H_SIDE_EFFECTS_SN_P          4792
H_SIDE_EFFECTS_A_P           4792
H_CLEAN_LINEAR_SCORE         4792
H_CLEAN_STAR_RATING          4792
H_QUIET_HSP_A_P              4792
H_QUIET_HSP_SN_P             4792
H_QUIET_HSP_U_P              4792
H_QUIET_LINEAR_SCORE         4792
H_QUIET_STAR_RATING          4792
H_HSP_RATING_0_6             4792
H_HSP_RATING_7_8             4792
H_HSP_RATING_9_10            4792
H_HSP_RATING_LINEAR_SCORE    4792
H_HSP_RATING_STAR_RATING     4792
H_RECMND_D

In [57]:
hcahps.groupby(

    ["Facility ID", "HCAHPS Measure ID"]

).size()

Facility ID  HCAHPS Measure ID   
010001       H_CLEAN_HSP_A_P         1
             H_CLEAN_HSP_SN_P        1
             H_CLEAN_HSP_U_P         1
             H_CLEAN_LINEAR_SCORE    1
             H_CLEAN_STAR_RATING     1
                                    ..
671303       H_SIDE_EFFECTS_SN_P     1
             H_SIDE_EFFECTS_U_P      1
             H_STAR_RATING           1
             H_SYMPTOMS_N_P          1
             H_SYMPTOMS_Y_P          1
Length: 325856, dtype: int64

In [58]:
drop_cols = [

    "Facility Name",

    "Address",

    "City/Town",

    "State",

    "ZIP Code",

    "County/Parish",

    "Telephone Number"

]
keep_core = [
    "Facility ID",
    "HCAHPS Measure ID",
    "HCAHPS Question",
    "HCAHPS Answer Description",
    "Patient Survey Star Rating",
    "HCAHPS Answer Percent",
    "HCAHPS Linear Mean Value",
    "Number of Completed Surveys",
    "Survey Response Rate Percent",
    "Start Date",
    "End Date"
]
footnote_cols = [
    "Patient Survey Star Rating Footnote",
    "HCAHPS Answer Percent Footnote",
    "Number of Completed Surveys Footnote",
    "Survey Response Rate Percent Footnote"
]

In [59]:
numeric_like = [

    "Patient Survey Star Rating",

    "HCAHPS Answer Percent",

    "HCAHPS Linear Mean Value",

    "Number of Completed Surveys",

    "Survey Response Rate Percent"

]

for col in numeric_like:

    print("\n", col)

    print(hcahps[col].value_counts(dropna=False).head(15))


 Patient Survey Star Rating
Patient Survey Star Rating
Not Applicable    282728
Not Available      14544
3                  10033
4                   8697
2                   5508
5                   3296
1                   1050
Name: count, dtype: int64

 HCAHPS Answer Percent
HCAHPS Answer Percent
Not Applicable    81464
Not Available     51932
4                  6526
5                  6212
6                  6004
3                  5792
7                  5211
17                 5105
18                 4991
19                 4910
11                 4765
8                  4763
20                 4761
16                 4759
13                 4663
Name: count, dtype: int64

 HCAHPS Linear Mean Value
HCAHPS Linear Mean Value
Not Applicable    287520
Not Available      12928
89                  1976
91                  1972
90                  1946
88                  1788
87                  1759
92                  1686
86                  1586
85                  1390
84       

In [60]:
hcahps_clean = hcahps[keep_core].copy()
for col in numeric_like:

    hcahps_clean[col] = pd.to_numeric(

        hcahps_clean[col],

        errors="coerce"

    )

In [61]:
hcahps_clean["Start Date"] = pd.to_datetime(

    hcahps_clean["Start Date"],

    errors="coerce"

)

hcahps_clean["End Date"] = pd.to_datetime(

    hcahps_clean["End Date"],

    errors="coerce"

)

In [63]:
measure_lookup = (

    hcahps_clean[

        ["HCAHPS Measure ID", "HCAHPS Question"]

    ]

    .drop_duplicates()

    .sort_values("HCAHPS Measure ID")

)

print(measure_lookup.to_string(index=False))

        HCAHPS Measure ID                                                                                                                    HCAHPS Question
          H_CLEAN_HSP_A_P                                                             Patients who reported that their room and bathroom were "Always" clean
         H_CLEAN_HSP_SN_P                                               Patients who reported that their room and bathroom were "Sometimes" or "Never" clean
          H_CLEAN_HSP_U_P                                                            Patients who reported that their room and bathroom were "Usually" clean
     H_CLEAN_LINEAR_SCORE                                                                                                    Cleanliness - linear mean score
      H_CLEAN_STAR_RATING                                                                                                          Cleanliness - star rating
             H_COMP_1_A_P                                 

In [64]:
selected_measure_ids = [

    "H_COMP_1_A_P",        # Nurses always communicated well

    "H_COMP_2_A_P",        # Doctors always communicated well

    "H_COMP_5_A_P",        # Staff always explained medicines

    "H_COMP_6_Y_P",        # Given information for recovery at home

    "H_CLEAN_HSP_A_P",     # Room/bathroom always clean

    "H_QUIET_HSP_A_P",     # Always quiet at night

    "H_HSP_RATING_9_10",   # Hospital rating 9 or 10

    "H_RECMND_DY"          # Definitely recommend hospital

]
hcahps_selected = hcahps_clean[
    hcahps_clean["HCAHPS Measure ID"].isin(selected_measure_ids)
].copy()

In [65]:
coverage = (
    hcahps_selected
    .groupby("HCAHPS Measure ID")["HCAHPS Answer Percent"]
    .agg(
        total_records="size",
        available_records="count"
    )
)

coverage["coverage_pct"] = (
    coverage["available_records"]
    / coverage["total_records"]
    * 100
)

print(coverage)

                   total_records  available_records  coverage_pct
HCAHPS Measure ID                                                
H_CLEAN_HSP_A_P             4792               3956     82.554257
H_COMP_1_A_P                4792               3956     82.554257
H_COMP_2_A_P                4792               3956     82.554257
H_COMP_5_A_P                4792               3956     82.554257
H_COMP_6_Y_P                4792               3956     82.554257
H_HSP_RATING_9_10           4792               3956     82.554257
H_QUIET_HSP_A_P             4792               3956     82.554257
H_RECMND_DY                 4792               3956     82.554257


In [66]:
missing_by_facility = (

    hcahps_selected

    .pivot(

        index="Facility ID",

        columns="HCAHPS Measure ID",

        values="HCAHPS Answer Percent"

    )

    .isna()

)

missing_by_facility.sum()

HCAHPS Measure ID
H_CLEAN_HSP_A_P      836
H_COMP_1_A_P         836
H_COMP_2_A_P         836
H_COMP_5_A_P         836
H_COMP_6_Y_P         836
H_HSP_RATING_9_10    836
H_QUIET_HSP_A_P      836
H_RECMND_DY          836
dtype: int64

In [67]:
missing_by_facility.sum(axis=1).value_counts()

0    3956
8     836
Name: count, dtype: int64

In [73]:
hcahps_meta = (
    hcahps_selected[
        [
            "Facility ID",
            "Number of Completed Surveys",
            "Survey Response Rate Percent",
            "Start Date",
            "End Date"
        ]
    ]
    .drop_duplicates(subset=["Facility ID"])
)
hcahps_wide = hcahps_selected.pivot(
    index="Facility ID",
    columns="HCAHPS Measure ID",
    values="HCAHPS Answer Percent"
).reset_index()
hcahps_wide = hcahps_wide.rename(columns={
    "H_COMP_1_A_P": "nurse_communication_pct",
    "H_COMP_2_A_P": "doctor_communication_pct",
    "H_COMP_5_A_P": "medication_communication_pct",
    "H_COMP_6_Y_P": "discharge_information_pct",
    "H_CLEAN_HSP_A_P": "cleanliness_pct",
    "H_QUIET_HSP_A_P": "quietness_pct",
    "H_HSP_RATING_9_10": "high_hospital_rating_pct",
    "H_RECMND_DY": "definitely_recommend_pct"
})
hcahps_final = hcahps_wide.merge(
    hcahps_meta,
    on="Facility ID",
    how="left"
)

In [75]:
hcahps_final.to_csv(

    "cleaned/hcahps_clean.csv",

    index=False

)

3. Medicare Spending Per Beneficiary

In [80]:
mspb = pd.read_csv("Medicare_Hospital_Spending_Per_Patient-Hospital.csv")
print(mspb.shape)
print(mspb.columns.tolist())
print(mspb.head())
print(mspb.dtypes)

(4628, 14)
['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Telephone Number', 'Measure ID', 'Measure Name', 'Score', 'Footnote', 'Start Date', 'End Date']
   Facility ID                    Facility Name                     Address  \
0        10001  SOUTHEAST HEALTH MEDICAL CENTER      1108 ROSS CLARK CIRCLE   
1        10005         MARSHALL MEDICAL CENTERS  2505 U S HIGHWAY 431 NORTH   
2        10006     NORTH ALABAMA MEDICAL CENTER         1701 VETERANS DRIVE   
3        10007         MIZELL MEMORIAL HOSPITAL               702 N MAIN ST   
4        10011               ST. VINCENT'S EAST  50 MEDICAL PARK EAST DRIVE   

    City/Town State  ZIP Code County/Parish Telephone Number Measure ID  \
0      DOTHAN    AL     36301       HOUSTON   (334) 793-8701     MSPB-1   
1        BOAZ    AL     35957      MARSHALL   (256) 593-8310     MSPB-1   
2    FLORENCE    AL     35630    LAUDERDALE   (256) 768-8400     MSPB-1   
3         OPP    AL   

In [85]:
# check data grain
mspb.groupby("Facility ID").size().value_counts()
mspb.duplicated(subset=["Facility ID"]).sum()

np.int64(0)

In [87]:
# change dtype
mspb["Facility ID"] = mspb["Facility ID"].astype("string")
mspb["ZIP Code"] = (

    mspb["ZIP Code"]

    .astype("string")

    .str.zfill(5)

)
mspb['Start Date'] = pd.to_datetime(mspb['Start Date'], errors='coerce')
mspb['End Date'] = pd.to_datetime(mspb['End Date'], errors='coerce')

In [93]:
keep_cols = [

    "Facility ID",

    "Measure ID",

    "Measure Name",

    "Score",

    "Footnote",

    "Start Date",

    "End Date"

]
mspb_clean = mspb[keep_cols].copy()

In [104]:
mspb_clean['Measure ID'].value_counts()
mspb_clean["Measure Name"].value_counts()
mspb_clean["Score"].value_counts()
mspb["Footnote"].value_counts()

Footnote
5.0     1505
19.0     170
1.0       73
29.0      51
Name: count, dtype: int64

In [ ]:
mspb_clean['Score'] = pd.to_numeric(mspb_clean['Score'], errors='coerce')

In [120]:
print(mspb_clean["Score"].describe())
print(mspb_clean["Score"].isna().mean())
print(mspb_clean["Footnote"].value_counts(dropna=False))

count    2880.000000
mean        0.990639
std         0.087723
min         0.540000
25%         0.940000
50%         0.990000
75%         1.040000
max         1.430000
Name: Score, dtype: float64
0.3777009507346586
Footnote
NaN     2829
5.0     1505
19.0     170
1.0       73
29.0      51
Name: count, dtype: int64


In [124]:
mspb_clean[
    mspb_clean["Score"].isna()
][["Score", "Footnote"]].value_counts(dropna=False)

Score  Footnote
NaN    5.0         1505
       19.0         170
       1.0           73
Name: count, dtype: int64

In [128]:
mspb_clean.to_csv(

    "cleaned/mspb_clean.csv",

    index=False

)

4. Unplanned Hospital Visits

In [3]:
uhv = pd.read_csv("Unplanned_Hospital_Visits-Hospital.csv")
print(uhv.shape)
print(uhv.columns.tolist())
print(uhv.head())
print(uhv.dtypes)

(67088, 20)
['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Telephone Number', 'Measure ID', 'Measure Name', 'Compared to National', 'Denominator', 'Score', 'Lower Estimate', 'Higher Estimate', 'Number of Patients', 'Number of Patients Returned', 'Footnote', 'Start Date', 'End Date']
  Facility ID                    Facility Name                 Address  \
0      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   
1      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   
2      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   
3      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   
4      010001  SOUTHEAST HEALTH MEDICAL CENTER  1108 ROSS CLARK CIRCLE   

  City/Town State  ZIP Code County/Parish Telephone Number   Measure ID  \
0    DOTHAN    AL     36301       HOUSTON   (334) 793-8701  EDAC_30_AMI   
1    DOTHAN    AL     36301       HOUSTON   (334) 793-8701   EDAC_30_HF 

/var/folders/d5/xf03rj855y5c5nqtgqmn3c380000gn/T/ipykernel_18097/773894390.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  uhv = pd.read_csv("Unplanned_Hospital_Visits-Hospital.csv")


In [8]:
#data grain check
uhv.groupby(['Facility ID', 'Measure ID']).size().value_counts()

1    67088
Name: count, dtype: int64

In [9]:
drop_redundant = [

    "Facility Name",

    "Address",

    "City/Town",

    "State",

    "ZIP Code",

    "County/Parish",

    "Telephone Number"

]

important_cols = [
    "Compared to National",
    "Denominator",
    "Score",
    "Lower Estimate",
    "Higher Estimate",
    "Number of Patients",
    "Number of Patients Returned",
    "Footnote"
]

for col in important_cols:
    print("\n---", col, "---")
    print(uhv[col].value_counts(dropna=False).head(20))


--- Compared to National ---
Compared to National
No Different Than the National Rate           23426
Not Available                                 15988
Number of Cases Too Small                     15026
Average Days per 100 Discharges                3713
More Days Than Average per 100 Discharges      2833
No Different than expected                     2586
Fewer Days Than Average per 100 Discharges     1789
Number of cases too small                       843
Worse Than the National Rate                    423
Better Than the National Rate                   326
Better than expected                             87
Worse than expected                              48
Name: count, dtype: int64

--- Denominator ---
Denominator
Not Available    31857
33                 259
31                 258
28                 256
25                 255
29                 254
39                 248
27                 247
35                 243
26                 238
32                 224
38           

In [10]:
numeric_cols = [

    "Denominator",

    "Score",

    "Lower Estimate",

    "Higher Estimate",

    "Number of Patients",

    "Number of Patients Returned"

]

for col in numeric_cols:

    uhv[col] = pd.to_numeric(uhv[col], errors="coerce")

In [11]:
for col in important_cols:
    print("\n---", col, "---")
    print(uhv[col].value_counts(dropna=False).head(20))


--- Compared to National ---
Compared to National
No Different Than the National Rate           23426
Not Available                                 15988
Number of Cases Too Small                     15026
Average Days per 100 Discharges                3713
More Days Than Average per 100 Discharges      2833
No Different than expected                     2586
Fewer Days Than Average per 100 Discharges     1789
Number of cases too small                       843
Worse Than the National Rate                    423
Better Than the National Rate                   326
Better than expected                             87
Worse than expected                              48
Name: count, dtype: int64

--- Denominator ---
Denominator
NaN     31857
33.0      259
31.0      258
28.0      256
25.0      255
29.0      254
39.0      248
27.0      247
35.0      243
26.0      238
32.0      224
38.0      223
40.0      223
30.0      219
44.0      217
43.0      215
41.0      213
36.0      212
34.0      202


In [12]:
uhv["Start Date"] = pd.to_datetime(

    uhv["Start Date"],

    errors="coerce"

)

uhv["End Date"] = pd.to_datetime(

    uhv["End Date"],

    errors="coerce"

)

In [13]:
uhv['Facility ID'] = uhv['Facility ID'].astype('string')

Facility ID                            object
Facility Name                          object
Address                                object
City/Town                              object
State                                  object
ZIP Code                                int64
County/Parish                          object
Telephone Number                       object
Measure ID                             object
Measure Name                           object
Compared to National                   object
Denominator                           float64
Score                                 float64
Lower Estimate                        float64
Higher Estimate                       float64
Number of Patients                    float64
Number of Patients Returned           float64
Footnote                               object
Start Date                     datetime64[ns]
End Date                       datetime64[ns]
dtype: object

In [29]:
# check missing rate
missing_rate = (
    uhv.groupby("Measure ID")[
        [
            "Score",
            "Denominator",
            "Number of Patients",
            "Number of Patients Returned"
        ]
    ]
    .apply(lambda x: x.isna().mean())
)

missing_rate

,Score,Denominator,Number of Patients,Number of Patients Returned
Measure ID,,,,
EDAC_30_AMI,0.675918,0.675918,0.675918,0.675918
EDAC_30_HF,0.342446,0.342446,0.342446,0.342446
EDAC_30_PN,0.242279,0.242279,0.242279,0.242279
Hybrid_HWR,0.121244,0.121244,1.000000,1.000000
OP_32,0.343698,0.343698,1.000000,1.000000
OP_35_ADM,0.685309,0.685309,1.000000,1.000000
OP_35_ED,0.685309,0.685309,1.000000,1.000000
OP_36,0.432179,0.432179,1.000000,1.000000
READM_30_AMI,0.619574,0.619574,1.000000,1.000000


In [31]:
uhv[["Measure ID", "Measure Name"]].drop_duplicates().sort_values("Measure ID")

,Measure ID,Measure Name
0,EDAC_30_AMI,Hospital return days for heart attack patients
1,EDAC_30_HF,Hospital return days for heart failure patients
2,EDAC_30_PN,Hospital return days for pneumonia patients
3,Hybrid_HWR,Hybrid Hospital-Wide All-Cause Readmission Mea...
4,OP_32,Rate of unplanned hospital visits after colono...
5,OP_35_ADM,Rate of inpatient admissions for patients rece...
6,OP_35_ED,Rate of emergency department (ED) visits for p...
7,OP_36,Ratio of unplanned hospital visits after hospi...
8,READM_30_AMI,Acute Myocardial Infarction (AMI) 30-Day Readm...
9,READM_30_CABG,Rate of readmission for CABG


In [32]:
selected_uhv_ids = [
    "Hybrid_HWR",        # hospital-wide readmission
    "READM_30_AMI",
    "READM_30_HF",
    "READM_30_COPD",
    "READM_30_PN",
    "READM_30_HIP_KNEE"
]

uhv_selected = uhv[

    uhv["Measure ID"].isin(selected_uhv_ids)

].copy()

In [34]:
uhv_selected.groupby("Measure ID")["Score"].apply(
    lambda x: x.notna().mean()
)

Measure ID
Hybrid_HWR           0.878756
READM_30_AMI         0.380426
READM_30_COPD        0.562604
READM_30_HF          0.657554
READM_30_HIP_KNEE    0.339107
READM_30_PN          0.757721
Name: Score, dtype: float64

In [20]:
uhv[important_cols].isna().mean()

Compared to National           0.000000
Denominator                    0.474854
Score                          0.474854
Lower Estimate                 0.474854
Higher Estimate                0.474854
Number of Patients             0.875760
Number of Patients Returned    0.875760
Footnote                       0.514190
dtype: float64

In [36]:


uhv_selected.groupby("Measure ID")["Compared to National"].value_counts(dropna=False)

Measure ID         Compared to National               
Hybrid_HWR         No Different Than the National Rate    3869
                   Not Available                           332
                   Number of Cases Too Small               249
                   Worse Than the National Rate            183
                   Better Than the National Rate           159
READM_30_AMI       No Different Than the National Rate    1806
                   Number of Cases Too Small              1754
                   Not Available                          1215
                   Worse Than the National Rate             11
                   Better Than the National Rate             6
READM_30_COPD      No Different Than the National Rate    2687
                   Number of Cases Too Small              1553
                   Not Available                           543
                   Worse Than the National Rate              9
READM_30_HF        No Different Than the National Rate    3038


In [37]:
uhv_keep = [
    "Facility ID",
    "Measure ID",
    "Measure Name",
    "Compared to National",
    "Denominator",
    "Score",
    "Lower Estimate",
    "Higher Estimate",
    "Footnote",
    "Start Date",
    "End Date"
]

uhv_final = uhv_selected[uhv_keep].copy()


In [38]:
print(
    uhv_final.duplicated(
        subset=["Facility ID", "Measure ID"]
    ).sum()
)

print(
    uhv_final.groupby("Measure ID")["Score"]
    .apply(lambda x: x.notna().mean())
)

print(uhv_final.dtypes)

0
Measure ID
Hybrid_HWR           0.878756
READM_30_AMI         0.380426
READM_30_COPD        0.562604
READM_30_HF          0.657554
READM_30_HIP_KNEE    0.339107
READM_30_PN          0.757721
Name: Score, dtype: float64
Facility ID                     object
Measure ID                      object
Measure Name                    object
Compared to National            object
Denominator                    float64
Score                          float64
Lower Estimate                 float64
Higher Estimate                float64
Footnote                        object
Start Date              datetime64[ns]
End Date                datetime64[ns]
dtype: object


In [39]:
uhv_final.groupby("Measure ID")["Score"].describe()

,count,mean,std,min,25%,50%,75%,max
Measure ID,,,,,,,,
Hybrid_HWR,4211.0,14.980052,0.707517,11.7,14.6,14.9,15.4,19.3
READM_30_AMI,1823.0,13.631267,0.831124,9.7,13.1,13.6,14.1,16.8
READM_30_COPD,2696.0,18.228746,0.856853,15.7,17.7,18.1,18.7,23.9
READM_30_HF,3151.0,19.708918,1.216124,14.3,19.0,19.6,20.4,26.6
READM_30_HIP_KNEE,1625.0,4.856062,0.707091,2.4,4.4,4.8,5.2,7.8
READM_30_PN,3631.0,15.991600,0.975537,12.7,15.4,15.9,16.5,22.7


In [40]:
uhv_final.to_csv(
    "cleaned/unplanned_visits_clean.csv",
    index=False
)

5. Census County Demographics

In [42]:
census = pd.read_csv("cc-est2025-agesex-all.csv", encoding="latin1")
print(census.shape)
print(census.columns.tolist())
print(census.head())

(22008, 96)
['SUMLEV', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME', 'YEAR', 'POPESTIMATE', 'POPEST_MALE', 'POPEST_FEM', 'UNDER5_TOT', 'UNDER5_MALE', 'UNDER5_FEM', 'AGE513_TOT', 'AGE513_MALE', 'AGE513_FEM', 'AGE1417_TOT', 'AGE1417_MALE', 'AGE1417_FEM', 'AGE1824_TOT', 'AGE1824_MALE', 'AGE1824_FEM', 'AGE16PLUS_TOT', 'AGE16PLUS_MALE', 'AGE16PLUS_FEM', 'AGE18PLUS_TOT', 'AGE18PLUS_MALE', 'AGE18PLUS_FEM', 'AGE1544_TOT', 'AGE1544_MALE', 'AGE1544_FEM', 'AGE2544_TOT', 'AGE2544_MALE', 'AGE2544_FEM', 'AGE4564_TOT', 'AGE4564_MALE', 'AGE4564_FEM', 'AGE65PLUS_TOT', 'AGE65PLUS_MALE', 'AGE65PLUS_FEM', 'AGE04_TOT', 'AGE04_MALE', 'AGE04_FEM', 'AGE59_TOT', 'AGE59_MALE', 'AGE59_FEM', 'AGE1014_TOT', 'AGE1014_MALE', 'AGE1014_FEM', 'AGE1519_TOT', 'AGE1519_MALE', 'AGE1519_FEM', 'AGE2024_TOT', 'AGE2024_MALE', 'AGE2024_FEM', 'AGE2529_TOT', 'AGE2529_MALE', 'AGE2529_FEM', 'AGE3034_TOT', 'AGE3034_MALE', 'AGE3034_FEM', 'AGE3539_TOT', 'AGE3539_MALE', 'AGE3539_FEM', 'AGE4044_TOT', 'AGE4044_MALE', 'AGE4044_FEM', 'AGE4549_TO

In [45]:
grain_check = (
    census.groupby(["STATE", "COUNTY", "YEAR"])
    .size()
)
print(grain_check.value_counts())

1    22008
Name: count, dtype: int64


In [46]:
census["YEAR"].value_counts().sort_index()

YEAR
1    3144
2    3144
3    3144
4    3144
5    3144
6    3144
7    3144
Name: count, dtype: int64

In [47]:
census["SUMLEV"].value_counts()

SUMLEV
50    22008
Name: count, dtype: int64

In [48]:
census_latest = census[census["YEAR"] == 7].copy()

In [54]:
census_latest.groupby(["STATE", "COUNTY"]).size().value_counts()

1    3144
Name: count, dtype: int64

In [55]:
census_latest["County_FIPS"] = (

    census_latest["STATE"].astype(str).str.zfill(2)

    + census_latest["COUNTY"].astype(str).str.zfill(3)

)

census_latest["County_FIPS"].duplicated().sum()

np.int64(0)

In [67]:
keep_cols = [
    "County_FIPS",
    "STATE",
    "COUNTY",
    "STNAME",
    "CTYNAME",
    "POPESTIMATE",
    "POPEST_MALE",
    "POPEST_FEM",
    "UNDER5_TOT",
    "AGE1824_TOT",
    "AGE2544_TOT",
    "AGE4564_TOT",
    "AGE65PLUS_TOT",
    "MEDIAN_AGE_TOT"
]

census_latest = census_latest[keep_cols].copy()

census_latest["Pct_65Plus"] = (
    census_latest["AGE65PLUS_TOT"] / census_latest["POPESTIMATE"] * 100
)

census_latest["Pct_Female"] = (
    census_latest["POPEST_FEM"] / census_latest["POPESTIMATE"] * 100
)

census_latest["Pct_Under5"] = (
    census_latest["UNDER5_TOT"] / census_latest["POPESTIMATE"] * 100
)

In [61]:
core_cols = [

    "POPESTIMATE",

    "POPEST_MALE",

    "POPEST_FEM",

    "UNDER5_TOT",

    "AGE1824_TOT",

    "AGE2544_TOT",

    "AGE4564_TOT",

    "AGE65PLUS_TOT",

    "MEDIAN_AGE_TOT"

]

print(census_latest['POPESTIMATE'].value_counts(dropna=False))

POPESTIMATE
5744     2
33894    2
2666     2
20118    2
41853    2
        ..
12901    1
25353    1
28669    1
9123     1
6840     1
Name: count, Length: 3092, dtype: int64


In [62]:
(census_latest["POPESTIMATE"] <= 0).sum()

np.int64(0)

In [64]:
(
    census_latest["POPEST_MALE"]
    + census_latest["POPEST_FEM"]
    - census_latest["POPESTIMATE"]
).value_counts()

0    3144
Name: count, dtype: int64

In [65]:
census_latest[core_cols].dtypes

POPESTIMATE         int64
POPEST_MALE         int64
POPEST_FEM          int64
UNDER5_TOT          int64
AGE1824_TOT         int64
AGE2544_TOT         int64
AGE4564_TOT         int64
AGE65PLUS_TOT       int64
MEDIAN_AGE_TOT    float64
dtype: object

In [69]:
census_latest.columns

Index(['County_FIPS', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME', 'POPESTIMATE',
       'POPEST_MALE', 'POPEST_FEM', 'UNDER5_TOT', 'AGE1824_TOT', 'AGE2544_TOT',
       'AGE4564_TOT', 'AGE65PLUS_TOT', 'MEDIAN_AGE_TOT', 'Pct_65Plus',
       'Pct_Female', 'Pct_Under5'],
      dtype='object')

In [122]:
[col for col in hospital.columns

 if "county" in col.lower()

 or "fips" in col.lower()

 or "state" in col.lower()]

['State', 'County/Parish']

#### merge hospital and census together through geographical info

In [124]:
print(hospital[["State", "County/Parish"]].head(10))
print(hospital["State"].unique()[:20])
print(hospital["County/Parish"].unique()[:20])


  State County/Parish
0    AL       HOUSTON
1    AL      MARSHALL
2    AL    LAUDERDALE
3    AL     COVINGTON
4    AL     JEFFERSON
5    AL       DE KALB
6    AL        SHELBY
7    AL     JEFFERSON
8    AL       COLBERT
9    AL          DALE
['AL' 'AK' 'AZ' 'AR' 'CA' 'CO' 'CT' 'DE' 'DC' 'FL' 'GA' 'HI' 'ID' 'IL'
 'IN' 'IA' 'KS' 'KY' 'LA' 'ME']
['HOUSTON' 'MARSHALL' 'LAUDERDALE' 'COVINGTON' 'JEFFERSON' 'DE KALB'
 'SHELBY' 'COLBERT' 'DALE' 'CHEROKEE' 'MONTGOMERY' 'LEE' 'ELMORE'
 'CULLMAN' 'MADISON' 'ETOWAH' 'MARION' 'FAYETTE' 'COFFEE' 'GREENE']


In [142]:
def clean_county_name(s):
    return (
        s.str.upper()
         .str.strip()
         .str.replace(r"\s+COUNTY$", "", regex=True)
         .str.replace(r"\s+PARISH$", "", regex=True)
         .str.replace(r"\s+CENSUS AREA$", "", regex=True)
    )

In [143]:
census_latest["County_Name_Clean"] = clean_county_name(
    census_latest["CTYNAME"]
)

hospital["County_Name_Clean"] = clean_county_name(
    hospital["County/Parish"]
)

In [144]:
import us
census_latest["State_Abbr"] = (
    census_latest["STNAME"]
    .apply(lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None))


In [145]:
census_latest[

    ["State_Abbr", "County_Name_Clean"]

].duplicated().sum()

np.int64(0)

In [174]:
county_fix = {
    ("AL", "DE KALB"): "DEKALB",
    ("AZ", "LAPAZ"): "LA PAZ",

    ("AK", "ANCHORAGE"): "ANCHORAGE MUNICIPALITY",
    ("AK", "MATANUSKA-SUSITNA"): "MATANUSKA-SUSITNA BOROUGH",
    ("AK", "JUNEAU"): "JUNEAU CITY AND BOROUGH",
    ("AK", "FAIRBANKS NORTH STAR"): "FAIRBANKS NORTH STAR BOROUGH",
    ("AK", "KENAI PENINSULA"): "KENAI PENINSULA BOROUGH",
    ("AK", "KETCHIKAN GATEWAY"): "KETCHIKAN GATEWAY BOROUGH",

    ("AK", "NORTHWEST ARTIC BOROUGH"): "NORTHWEST ARCTIC BOROUGH",
    ("AK", "NORTH SLOPE BOROUH"): "NORTH SLOPE BOROUGH",
}
county_fix.update({
    ("DC", "THE DISTRICT"): "DISTRICT OF COLUMBIA",

    ("FL", "DE SOTO"): "DESOTO",

    ("GA", "DE KALB"): "DEKALB",
    ("GA", "MC DUFFIE"): "MCDUFFIE",

    ("IL", "LA SALLE"): "LASALLE",
    ("IL", "MC DONOUGH"): "MCDONOUGH",
    ("IL", "MC HENRY"): "MCHENRY",
    ("IL", "DU PAGE"): "DUPAGE",
    ("IL", "MC LEAN"): "MCLEAN",
    ("IL", "DE KALB"): "DEKALB",
})
county_fix.update({
    ("AK", "SITKA BOROUGH"): "SITKA CITY AND BOROUGH",

    ("LA", "E. BATON ROUGE"): "EAST BATON ROUGE",
    ("LA", "JEFFRSON DAVIS"): "JEFFERSON DAVIS",
    ("LA", "ST. JOHN BAPTIST"): "ST JOHN THE BAPTIST",

    ("MD", "PRINCE GEORGES"): "PRINCE GEORGE'S",
    ("MD", "ST. MARYS"): "ST MARY'S",

    ("MN", "LAKE OF WOODS"): "LAKE OF THE WOODS",
    ("MN", "YELLOW MEDCINE"): "YELLOW MEDICINE",

    ("NE", "SCOTT BLUFF"): "SCOTTS BLUFF",
})
hospital["County_Name_Clean"] = hospital.apply(
    lambda row: county_fix.get(
        (row["State"], row["County_Name_Clean"]),
        row["County_Name_Clean"]
    ),
    axis=1
)

In [175]:
def make_match_key(s):

    return (

        s.str.upper()

         .str.strip()

         .str.replace(r"[^A-Z0-9]", "", regex=True)

    )

hospital["County_Match_Key"] = make_match_key(

    hospital["County_Name_Clean"]

)

census_latest["County_Match_Key"] = make_match_key(

    census_latest["County_Name_Clean"]

)

In [176]:
census_latest[

    ["State_Abbr", "County_Match_Key"]

].duplicated().sum()

np.int64(0)

In [177]:
census_key = census_latest[
    ["State_Abbr", "County_Match_Key", "County_FIPS"]
]

test_merge = hospital.merge(
    census_key,
    left_on=["State", "County_Match_Key"],
    right_on=["State_Abbr", "County_Match_Key"],
    how="left",
    validate="many_to_one"
)

test_merge["County_FIPS"].notna().mean()

np.float64(0.9779086892488954)

In [178]:
test_merge.loc[
    test_merge['County_FIPS'].isna(),
    ["State", "County/Parish"]
].drop_duplicates().head(20)

,State,County/Parish
110,AK,VALDEZ-CORDOVA
798,CT,HARTFORD
799,CT,WINDHAM
800,CT,LITCHFIELD
801,CT,NEW HAVEN
802,CT,FAIRFIELD
803,CT,NEW LONDON
804,CT,TOLLAND
810,CT,MIDDLESEX
847,DC,THE DISTRICT


In [179]:
unmatched = (
    test_merge.loc[
        test_merge["County_FIPS"].isna(),
        ["State", "County/Parish", "County_Match_Key"]
    ]
    .drop_duplicates()
    .sort_values(["State", "County/Parish"])
)

unmatched["State"].value_counts()

State
PR    24
CT     8
VI     2
AK     1
AS     1
DC     1
GU     1
MN     1
MP     1
NM     1
Name: count, dtype: int64

In [180]:
pd.set_option("display.max_rows", None)

unmatched.sort_values(["State", "County/Parish"])

,State,County/Parish,County_Match_Key
110,AK,VALDEZ-CORDOVA,VALDEZCORDOVA
5343,AS,EASTERN DISTRICT,EASTERNDISTRICT
802,CT,FAIRFIELD,FAIRFIELD
798,CT,HARTFORD,HARTFORD
800,CT,LITCHFIELD,LITCHFIELD
810,CT,MIDDLESEX,MIDDLESEX
801,CT,NEW HAVEN,NEWHAVEN
803,CT,NEW LONDON,NEWLONDON
804,CT,TOLLAND,TOLLAND
799,CT,WINDHAM,WINDHAM


In [ ]:
hospital_census = hospital.merge(

    census_latest,

    left_on=["State", "County_Match_Key"],

    right_on=["State_Abbr", "County_Match_Key"],

    how="left",

    validate="many_to_one"

)

hospital_census["Census_Matched"] = (

    hospital_census["County_FIPS"].notna()

)

hospital_census["Census_Matched"].value_counts()

Census_Matched
True     5312
False     120
Name: count, dtype: int64

In [183]:
unmatched_final = (

    hospital_census.loc[

        ~hospital_census["Census_Matched"],

        ["Facility ID", "Facility Name", "State", "County/Parish"]

    ]

    .drop_duplicates()

)

In [ ]:
# validation after merge
print("Before merge:", len(hospital))
print("After merge:", len(hospital_census))

print(
    "Unique facilities:",
    hospital_census["Facility ID"].nunique()
)

print(
    "Duplicate facility IDs:",
    hospital_census["Facility ID"].duplicated().sum()
)

print(
    "Census match rate:",
    hospital_census["Census_Matched"].mean()
)

Before merge: 5432
After merge: 5432
Unique facilities: 5432
Duplicate facility IDs: 0
Census match rate: 0.9779086892488954


#### create useful columns

In [185]:
county_hospital_count = (
    hospital_census[
        hospital_census["Census_Matched"]
    ]
    .groupby("County_FIPS")["Facility ID"]
    .nunique()
    .rename("Hospital_Count")
)
county_hospital_count

County_FIPS
01001     1
01003     4
01005     1
01007     1
01009     1
01011     2
01013     1
01015     1
01017     1
01019     1
01021     1
01023     1
01025     2
01027     1
01031     1
01033     2
01035     1
01039     2
01041     2
01043     2
01045     1
01047     1
01049     1
01051     2
01053     2
01055     3
01057     2
01059     2
01061     2
01063     1
01065     1
01069     2
01071     1
01073    11
01077     1
01081     1
01083     1
01089     3
01091     1
01093     1
01095     1
01097     6
01099     1
01101     4
01103     1
01109     1
01111     1
01113     1
01115     1
01117     1
01119     1
01121     2
01123     2
01125     4
01127     1
01129     1
01131     1
01133     1
02020     6
02050     1
02070     1
02090     2
02110     1
02122     3
02130     1
02150     1
02170     1
02180     1
02185     1
02188     1
02195     1
02220     1
02275     1
04001     4
04003     4
04005     4
04007     3
04009     1
04012     2
04013    53
04015     5
04017     6
0401

In [192]:
pd.reset_option("display.max_rows")
hospital_census = hospital_census.merge(
    county_hospital_count,
    on='County_FIPS',
    how='left',
    validate="many_to_one"
)

In [193]:
hospital_census["Hospitals_per_100k"] = (

    hospital_census["Hospital_Count"]

    / hospital_census["POPESTIMATE"]

    * 100000

)

In [198]:
county_summary = (

    hospital_census[

        [

            "County_FIPS",

            "STNAME",

            "CTYNAME",

            "POPESTIMATE",

            "Hospital_Count",

            "Hospitals_per_100k"

        ]

    ]

    .drop_duplicates("County_FIPS")

)
county_summary

,County_FIPS,STNAME,CTYNAME,POPESTIMATE,Hospital_Count,Hospitals_per_100k
0,01069,Alabama,Houston County,110318.0,2.0,1.812941
1,01095,Alabama,Marshall County,103537.0,1.0,0.965838
2,01077,Alabama,Lauderdale County,97135.0,1.0,1.029495
3,01039,Alabama,Covington County,37947.0,2.0,5.270509
4,01073,Alabama,Jefferson County,665742.0,11.0,1.652292
...,...,...,...,...,...,...
5426,48253,Texas,Jones County,20861.0,1.0,4.793634
5427,48405,Texas,San Augustine County,7722.0,1.0,12.950013
5429,48211,Texas,Hemphill County,3083.0,1.0,32.435939
5430,48429,Texas,Stephens County,9380.0,1.0,10.660981


In [199]:
hospital_census.to_csv(

    "cleaned/hospital_census_clean.csv",

    index=False

)

In [202]:
county_summary = (
    hospital_census[
        [
            "County_FIPS",
            "STNAME",
            "CTYNAME",
            "POPESTIMATE",
            "Pct_65Plus",
            "Pct_Female",
            "Pct_Under5",
            "MEDIAN_AGE_TOT",
            "Hospital_Count",
            "Hospitals_per_100k"
        ]
    ]
    .drop_duplicates("County_FIPS")
)

In [205]:
county_summary.to_csv(
    "cleaned/county_summary_clean.csv",
    index=False
)